In [0]:
-- build country import export summary
DROP TABLE silver_country_trade_summary;
CREATE TABLE silver_country_trade_summary
WITH exports as(
SELECT source_country_code_iso_2 as country_code, source_country_name as country_name, sum(primary_value) as exports from silver_trade_flow GROUP BY source_country_code_iso_2, source_country_name),
imports as(
SELECT target_country_code_iso_2 as country_code, target_country_name as country_name, sum(primary_value) as imports from silver_trade_flow GROUP BY target_country_code_iso_2, target_country_name
)
SELECT i.*, e.exports FROM exports as e JOIN imports as i ON e.country_code = i.country_code
ORDER BY exports + imports DESC;

DROP TABLE IF EXISTS silver_country_trade_weight_summary;
CREATE TABLE silver_country_trade_weight_summary
WITH exports as(
SELECT source_country_code_iso_2 as country_code, source_country_name as country_name, sum(net_weight) as export_weight from silver_trade_flow GROUP BY source_country_code_iso_2, source_country_name),
imports as(
SELECT target_country_code_iso_2 as country_code, target_country_name as country_name, sum(net_weight) as import_weight from silver_trade_flow GROUP BY target_country_code_iso_2, target_country_name
)
SELECT i.*, e.export_weight FROM exports as e JOIN imports as i ON e.country_code = i.country_code
ORDER BY export_weight + import_weight DESC;

-- build imports
DROP TABLE silver_country_imports;
CREATE TABLE silver_country_imports
with total_imports as 
(SELECT hs_code, target_country_code_iso_2 as country_code, target_country_name as country, sum(primary_value) as total_imports 
FROM silver_trade_flow GROUP BY hs_code, target_country_code_iso_2, target_country_name)
SELECT t.hs_code, t.target_country_code_iso_2 as country_code, t.target_country_name as country, t.source_country_code_iso_2 as source_country_code, t.source_country_name as source_country_name, t.primary_value as imports, total_imports.total_imports, t.primary_value/total_imports.total_imports as import_share
FROM silver_trade_flow as t LEFT JOIN total_imports ON t.hs_code = total_imports.hs_code AND t.target_country_code_iso_2 = total_imports.country_code
WHERE t.primary_value IS NOT NULL
ORDER BY hs_code, country_code, import_share DESC;

SELECT * FROM silver_country_imports
ORDER BY hs_code, country_code, import_share

-- Build reference table
DROP TABLE silver_country_reference;
CREATE TABLE silver_country_reference (
    country_code string, 
    country string,
    status string
);
INSERT INTO silver_country_reference
VALUES(
    'HK', 'China, Hong Kong (SAR)','hub'
),
(
    'SG', 'Singapore', 'hub'
),
(
    'AE', 'United Arab Emirates', 'hub'
),
(
    'NL', 'Netherlands', 'hub'
);


-- build hyperparameter tables
DECLARE VARIABLE k_control DOUBLE;
SET VARIABLE k_control = 1;
DROP TABLE silver_country_hyperparamers;
CREATE TABLE silver_country_hyperparamers 
SELECT 
s.country_code, 
s.country_name, 
s.imports/(s.imports+s.exports) as r, 
GREATEST(0, 1- s.exports/s.imports) as theta_floor,  
 -- implement hard coded overrides to hyper parameter tables
CASE WHEN ref.country_code IS NOT NULL AND ref.status = 'hub' THEN 1 ELSE 1+ k_control*r END as a, 
CASE WHEN ref.country_code IS NOT NULL AND ref.status = 'hub' THEN 4 ELSE 1 + k_control*(1-r) END as b 
FROM silver_country_trade_summary as s
LEFT JOIN silver_country_reference as ref on ref.country_code = s.country_code;

-- build hyperparameters that are hs_code specific

-- build country_hs_code import export summary
DROP TABLE silver_country_hs_code_trade_summary;
CREATE TABLE silver_country_hs_code_trade_summary
WITH exports as(
SELECT hs_code, source_country_code_iso_2 as country_code, source_country_name as country_name, sum(primary_value) as exports from silver_trade_flow GROUP BY hs_code, source_country_code_iso_2, source_country_name),
imports as(
SELECT hs_code, target_country_code_iso_2 as country_code, target_country_name as country_name, sum(primary_value) as imports from silver_trade_flow GROUP BY hs_code, target_country_code_iso_2, target_country_name
)
SELECT e.*, i.imports, e.exports/i.imports as relative_exports FROM exports as e LEFT JOIN imports as i ON e.country_code = i.country_code AND e.hs_code=i.hs_code
ORDER BY exports + imports DESC;

DROP TABLE IF EXISTS silver_country_hs_code_trade_weight_summary;
CREATE TABLE silver_country_hs_code_trade_weight_summary
WITH exports as(
SELECT hs_code, source_country_code_iso_2 as country_code, source_country_name as country_name, sum(net_weight) as export_weight from silver_trade_flow GROUP BY hs_code, source_country_code_iso_2, source_country_name),
imports as(
SELECT hs_code, target_country_code_iso_2 as country_code, target_country_name as country_name, sum(net_weight) as import_weight from silver_trade_flow GROUP BY hs_code, target_country_code_iso_2, target_country_name
)
SELECT e.*, i.import_weight FROM exports as e LEFT JOIN imports as i ON e.country_code = i.country_code AND e.hs_code=i.hs_code
ORDER BY export_weight + import_weight DESC;




SELECT * FROM silver_country_hs_code_trade_weight_summary WHERE country_code = 'AU' AND hs_code = '8703'

-- build hyperparameter tables
DECLARE VARIABLE k_control DOUBLE;
SET VARIABLE k_control = 10;
DROP VARIABLE epsilon;
DECLARE VARIABLE epsilon DOUBLE;
SET VARIABLE epsilon = 0.1;
DROP TABLE IF EXISTS silver_country_hs_code_hyperparamers_2;
CREATE TABLE silver_country_hs_code_hyperparamers_2 
SELECT 
s.hs_code,
s.country_code, 
s.country_name,
s.exports,
s.imports,
s.exports/(s.imports+s.exports) as r, 
 -- implement hard coded overrides to hyper parameter tables
epsilon+ k_control*r as a, 
epsilon + k_control*(1-r)  as b 
FROM silver_country_hs_code_trade_summary as s
LEFT JOIN silver_country_reference as ref on ref.country_code = s.country_code;

SELECT * FROM silver_country_hs_code_hyperparamers_2 WHERE country_code = 'CD' AND hs_code = '8703'

-- build hyperparameter tables (v3: mean-preserving reparameterization)
DECLARE OR REPLACE VARIABLE nu_control DOUBLE DEFAULT 2;
DECLARE OR REPLACE VARIABLE epsilon DOUBLE DEFAULT 0.001;

DROP TABLE IF EXISTS silver_country_hs_code_hyperparamers_3;
CREATE TABLE silver_country_hs_code_hyperparamers_3
SELECT
    s.hs_code,
    s.country_code,
    s.country_name,
    s.exports,
    s.imports,
    s.exports / (s.imports + s.exports) AS r,
    -- mu is r, clipped away from the exact 0/1 boundary so a,b stay strictly positive
    LEAST(GREATEST(s.exports / (s.imports + s.exports), epsilon), 1 - epsilon) AS mu,
    nu_control AS nu,
    -- a/(a+b) = mu exactly, regardless of nu — mean and spread are now decoupled
    LEAST(GREATEST(s.exports / (s.imports + s.exports), epsilon), 1 - epsilon) * nu_control AS a,
    (1 - LEAST(GREATEST(s.exports / (s.imports + s.exports), epsilon), 1 - epsilon)) * nu_control AS b
FROM silver_country_hs_code_trade_summary AS s
LEFT JOIN silver_country_reference AS ref ON ref.country_code = s.country_code;

DECLARE OR REPLACE VARIABLE nu_base DOUBLE DEFAULT 2;        -- fixes uniform exactly at r=0.5
DECLARE OR REPLACE VARIABLE k_spread DOUBLE DEFAULT 20;      -- max extra concentration at full imbalance
DECLARE OR REPLACE VARIABLE gamma DOUBLE DEFAULT 4;          -- speed control: higher = slower to react near parity
DECLARE OR REPLACE VARIABLE epsilon DOUBLE DEFAULT 0.001;

DROP TABLE IF EXISTS silver_country_hs_code_hyperparamers_4;
CREATE TABLE silver_country_hs_code_hyperparamers_4 AS
WITH base AS (
    SELECT
        s.hs_code, s.country_code, s.country_name, s.exports, s.imports,
        s.exports / (s.imports + s.exports) AS r
    FROM silver_country_hs_code_trade_summary AS s
    LEFT JOIN silver_country_reference AS ref ON ref.country_code = s.country_code
),
shaped AS (
    SELECT *,
        2*r - 1 AS d,
        SIGN(2*r - 1) * POWER(ABS(2*r - 1), gamma) AS d_shaped
    FROM base
)
SELECT
    hs_code, country_code, country_name, exports, imports, r,
    LEAST(GREATEST(0.5 + 0.5*d_shaped, epsilon), 1 - epsilon) AS mu,
    nu_base + k_spread * ABS(d_shaped) AS nu,
    LEAST(GREATEST(0.5 + 0.5*d_shaped, epsilon), 1 - epsilon) * (nu_base + k_spread*ABS(d_shaped)) AS a,
    (1 - LEAST(GREATEST(0.5 + 0.5*d_shaped, epsilon), 1 - epsilon)) * (nu_base + k_spread*ABS(d_shaped)) AS b
FROM shaped;

SELECT * FROM silver_country_hs_code_hyperparamers_4 WHERE country_code = 'TW' ORDER BY exports desc;

SELECT * FROM silver_country_hs_code_trade_summary WHERE hs_code in ('7308', '7207', '2601') AND country_code = 'JP' ORDER BY hs_code;

DROP TABLE IF EXISTS silve_import_export_analysis;
CREATE TABLE silve_import_export_analysis AS
SELECT *
FROM (
    SELECT
        hs_code,
        country_code,
        country_name,
        exports,
        imports,
        relative_exports,
        exports / MAX(exports) OVER (PARTITION BY hs_code) AS exports_share_of_max,
        imports / MAX(imports) OVER (PARTITION BY hs_code) AS imports_share_of_max,
        exports / AVG(exports) OVER (PARTITION BY hs_code) AS exports_vs_avg_country,
        imports / AVG(imports) OVER (PARTITION BY hs_code) AS imports_vs_avg_country
    FROM silver_country_hs_code_trade_summary
) t
ORDER BY hs_code;
SELECT * FROM silve_import_export_analysis WHERE hs_code in ('2601', '7207', '7208', '7209', '7210', '7225', '7226', '8708', '8703') AND country_code = 'JP' ORDER BY hs_code;

SELECT * FROM silve_import_export_analysis WHERE hs_code in ('1806', '1803', '1805', '1804', '1801') AND country_code = 'CA' ORDER BY hs_code;

SELECT s.*
FROM silve_import_export_analysis AS s
JOIN (
    VALUES
        ('AE', '7408'),
        ('AE', '7403'),
        ('AE', '2603'),
        ('CL', '2603'),
        ('NL', '1806'),
        ('NL', '1803'),
        ('NL', '1801'),
        ('DE', '1801'),
        ('KR', '7207'),
        ('JP', '8703'),
        ('JP', '8707'),
        ('JP', '7210'),
        ('AU', '2601'),
        ('AU', '2603'),
        ('CN', '5205'),
        ('CN', '5201'),
        ('US', '5201'),
        ('IN', '5201'),
        ('IN', '5205'),
        ('AU', '5201')
) AS benchmark (country_code, hs_code)
ON s.country_code = benchmark.country_code
AND s.hs_code = benchmark.hs_code;

SELECT s.*, h.r, h.mu, h.nu, h.a, h.b
FROM silve_import_export_analysis AS s
JOIN (
    VALUES
        ('NL', '1806'),
        ('NL', '1803'),
        ('NL', '1801')
) AS benchmark (country_code, hs_code)
ON s.country_code = benchmark.country_code
AND s.hs_code = benchmark.hs_code
LEFT JOIN silver_country_hs_code_hyperparamers_4 AS h
ON s.country_code = h.country_code
AND s.hs_code = h.hs_code
ORDER BY s.country_code

SELECT s.*, h.r, h.mu, h.nu, h.a, h.b
FROM silve_import_export_analysis AS s
JOIN (
    VALUES
        ('AE', '7408'),
        ('AE', '7403'),
        ('AE', '2603')
) AS benchmark (country_code, hs_code)
ON s.country_code = benchmark.country_code
AND s.hs_code = benchmark.hs_code
LEFT JOIN silver_country_hs_code_hyperparamers_4 AS h
ON s.country_code = h.country_code
AND s.hs_code = h.hs_code
ORDER BY s.country_code


SELECT * FROM bronze_trade_flows
 

SELECT date, hs_code, source_country_code_iso_2, sum(net_weight) as weight FROM silver_trade_flow WHERE hs_code = 2610 GROUP BY date, hs_code, source_country_code_iso_2 order by weight desc;
